# Модель задачи 1: предсказание оценки

на вход берём то что собрал 03_task1_dataset.ipynb. task1_embed_inputs.parquet это индексы категорий с паддингом до 10 и 15 масштабированных числовых признаков. task1_features.parquet это циклические временные признаки. task1_dataset.parquet это ключи, даты и сырые агрегаты, они нужны для prior-признаков, сегментов и бейзлайнов. ещё есть словари индексов task1_vocab_*.json, их фитили только на train.

предсказываем оценку stars от 1 до 5, которую юзер поставит заведению, по табличным признакам юзера и заведения. id юзера и заведения в основной модели не используем, почему так разбираем ниже. отдельно проверяем эмбеддинг заведения в абляции.

## что диктуют данные

срез нарезан по времени 70/15/15, train это до начала 2019, test после апреля 2020, то есть оцениваемся на будущем и агрегаты не должны подсматривать вперёд. в тесте 62.6% строк это юзеры которых не было в train, медиана отзывов на юзера равна одному, так что эмбеддинг id юзера почти везде был бы случайным вектором для незнакомого значения. ещё 20% строк теста это новые заведения, их эмбеддинг тоже промахнётся. таргет дрейфует, в train 43% пятёрок и 14% единиц, в test 53% и 22%, оценки поляризованы и тест объективно труднее. categories у заведения это мультилейбл, несколько кухонь сразу, поэтому категории удобнее пулить эмбеддингом а не делать one-hot.

главный вывод отсюда, модель опирается только на признаки, без id юзера и заведения. для холодного старта это обычная рекомендация, без истории взаимодействий работают модели на атрибутах, см. [обзор cold-start](https://arxiv.org/pdf/2501.01945) и [wikipedia](https://en.wikipedia.org/wiki/Cold_start_(recommender_systems)).

## бейзлайны как планка

ниже считаем на этом же сплите глобальное среднее, среднюю заведения, среднюю юзера, простой бейзлайн из суммы средней юзера и средней заведения минус глобальное среднее (это классика со времён [Netflix Prize](https://datajobs.com/data-science-repo/Recommender-Systems-%5BNetflix%5D.pdf)) и ridge по всем числовым признакам. ridge выходит не лучше суммы двух средних, около 1.12 RMSE, то есть линейная комбинация признаков ничего не добавляет. значит сеть оправдана только если бьёт этот бейзлайн, а выигрыш может прийти лишь из нелинейностей, взаимодействий признаков и более информативных фич.

## что взяли из литературы

категориальные признаки кодируем обучаемыми эмбеддингами, это компактнее one-hot и лучше работает на разреженных данных, [Guo & Berkhahn 2016](https://arxiv.org/abs/1604.06737). мультилейбл categories пропускаем через EmbeddingBag, то есть усредняем эмбеддинги кухонь заведения. дальше, MLP плохо выучивает скалярное произведение из конкатенации, это основной результат [Rendle et al. 2020](https://dl.acm.org/doi/10.1145/3383313.3412488), поэтому мультипликативные взаимодействия мы подаём сети явно, скалярное и поэлементное произведение векторов юзера и заведения. сама компоновка, нижние сети для плотных признаков, эмбеддинги для разреженных, слой взаимодействий и верхняя сеть, повторяет [DLRM](https://arxiv.org/abs/1906.00091) и идёт в русле Wide&Deep, DeepFM и DCN. размер эмбеддинга категории стартует с 16, увеличение до 32 проверяем дальше. и наконец холодный старт лечится именно признаками, юзер описан агрегатами, заведение агрегатами и категориями.

## prior-признаки по прошлому

статические агрегаты average_stars и biz_avg_stars усреднены по всей истории, в них мягкая утечка, они включают будущие отзывы. поэтому добавляем их честные версии, посчитанные строго по прошлому относительно даты каждого отзыва. для юзера это средняя оценка, число отзывов и давность последнего отзыва на момент текущего, то же для заведения. идея что у baseline-предикторов есть временная динамика и её учёт даёт основной прирост идёт от Netflix Prize, [Koren KDD 2009](https://dl.acm.org/doi/10.1145/1557019.1557072). утечки нет по построению, для каждой строки берём только более ранние отзывы, ниже это ещё проверяем вручную. гео-дистанцию и ценовую близость тоже считаем, но в абляции они прироста не дали.

## архитектура

юзерная ветка это маленькая сеть над числовыми и prior-признаками юзера. ветка заведения берёт числовые и prior-признаки заведения, эмбеддинг категорий через EmbeddingBag и one-hot города. два вектора, юзера и заведения, мы соединяем вместе с их поэлементным и скалярным произведением и контекстом времени, и подаём в верхнюю сеть с BatchNorm, ReLU и Dropout. голова два варианта, регрессия одним нейроном и классификация на 5 классов, их сравниваем дальше. для сравнения обучаем ещё плоскую сеть на той же конкатенации без явного взаимодействия.

## что отвергли

чистые MF и NCF на id не работают при 62.6% холодных юзеров в тесте. эмбеддинг id юзера мёртв, медиана отзывов на юзера это один. эмбеддинг заведения проверяем абляцией при фиксированном лоссе. TabTransformer и AutoInt окупаются на десятках категориальных полей и миллиардах примеров, у нас три категориальных поля и 465 тысяч строк. two-tower с чистым скалярным произведением нужен для поиска по миллионам айтемов, при 17.5 тысяч заведений можно скорить всё напрямую. больше данных тоже не помогает, кривая обучения выходит на плато.

## таргет, голова и лосс

таргет это stars от 1 до 5. базовая голова регрессия одним нейроном с клипом в диапазон от 1 до 5 на инференсе, сравниваем два лосса, MSE который точно отвечает метрике RMSE, и Huber который меньше штрафует большие неизбежные промахи на единицах. вторая голова это классификация на 5 классов с кросс-энтропией, а предсказание считаем как матожидание softmax, сумму вероятностей на номер оценки. приём известен как DEX, [Rothe et al. 2015](https://www.cv-foundation.org/openaccess/content_iccv_2015_workshops/w11/html/Rothe_DEX_Deep_EXpectation_ICCV_2015_paper.html), рядом стоит порядковая регрессия [CORAL](https://arxiv.org/abs/1901.07884). работает потому что оценки дискретны и идут либо к единице либо к пятёрке, softmax описывает всё распределение а не только среднее. год в признаки не берём, train заканчивается 2019-м и экстраполяция года на ковидный тест только мешает, циклическое время через sin и cos берём.

## как сравниваем

все сравнения архитектура против архитектуры и абляции делаем при фиксированном лоссе и голове. лучшую конфигурацию выбираем по val, метрики на тесте печатаем один раз в разделе результатов, чтобы тест не утекал в выбор гипотез. мелкие дельты меньше 0.01 RMSE гоняем на трёх сидах и смотрим среднее с разбросом. одиночную модель для сравнения со средним из трёх берём на дефолтном сиде, сид по val не подбираем.

## импорты и настройки

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
from _constants import PROCESSED, ARTIFACTS, ENABLE_LOGGING

In [ ]:
SEED = 42
SEEDS_MULTI = [42, 7, 2024]
MAXLEN = 10
D_CAT = 16
D_BRANCH = 32
D_BIZ_EMB = 32
N_CITY = 4
BATCH = 1024
MAX_EPOCHS = 30
PATIENCE = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.2
USER_NUM = ["num_user_review_count", "num_average_stars", "num_user_useful", "num_user_funny", "num_user_cool", "num_fans", "num_n_friends", "num_n_elite_years", "num_account_age_days"]
BIZ_NUM = ["num_biz_avg_stars", "num_biz_review_count", "num_is_open", "num_price_range", "num_price_known"]
CROSS = ["num_user_avg_minus_biz"]
TIME_COLS = ["month_sin", "month_cos", "dow_sin", "dow_cos", "hour_sin", "hour_cos", "is_weekend"]
USER_PRIOR = ["u_prior_mean", "u_cnt", "u_days_prev", "u_has_hist"]
BIZ_PRIOR = ["b_prior_mean", "b_cnt", "b_days_prev", "b_has_hist"]
CROSS_PRIOR = ["u_geo_dist", "price_aff"]
CATID_COLS = [f"catid_{i}" for i in range(MAXLEN)]
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device {device}")

## prior-признаки строго по прошлому

сортируем все отзывы по дате и для каждой строки считаем expanding-статистики юзера и заведения без текущего отзыва, через cumsum минус текущее значение. для каждой строки берём только более ранние отзывы, утечки нет по построению, ниже это проверяем ручной сверкой. гео-дистанцию и ценовую близость тоже считаем, они участвуют в абляции.

In [ ]:
cols = ["review_id", "user_id", "business_id", "stars", "date", "latitude", "longitude", "price_range", "split"]
ds = pd.read_parquet(PROCESSED / "task1_dataset.parquet", columns=cols)
ds["date"] = pd.to_datetime(ds["date"])
ds = ds.sort_values(["date", "review_id"], kind="mergesort").reset_index(drop=True)
ds.shape

In [ ]:
gu = ds.groupby("user_id", sort=False)
ds["u_cnt"] = gu.cumcount()
ds["u_prior_mean"] = np.where(ds.u_cnt > 0, (gu["stars"].cumsum() - ds["stars"]) / ds.u_cnt, np.nan)
ds["u_days_prev"] = gu["date"].diff().dt.total_seconds() / 86400

In [ ]:
for c in ["latitude", "longitude"]:
    ds[f"u_pc_{c}"] = np.where(ds.u_cnt > 0, (gu[c].cumsum() - ds[c]) / ds.u_cnt, np.nan)
lat1, lon1 = np.radians(ds.latitude), np.radians(ds.longitude)
lat2, lon2 = np.radians(ds.u_pc_latitude), np.radians(ds.u_pc_longitude)
h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
ds["u_geo_dist"] = 2 * 6371 * np.arcsin(np.sqrt(h))
ds["u_prior_price"] = np.where(ds.u_cnt > 0, (gu["price_range"].cumsum() - ds["price_range"]) / ds.u_cnt, np.nan)
ds["price_aff"] = ds["u_prior_price"] - ds["price_range"]

In [ ]:
gb = ds.groupby("business_id", sort=False)
ds["b_cnt"] = gb.cumcount()
ds["b_prior_mean"] = np.where(ds.b_cnt > 0, (gb["stars"].cumsum() - ds["stars"]) / ds.b_cnt, np.nan)
ds["b_days_prev"] = gb["date"].diff().dt.total_seconds() / 86400

In [ ]:
u = ds[ds.u_cnt >= 3]["user_id"].iloc[0]
sub = ds[ds.user_id == u].sort_values("date")
for k in range(1, min(4, len(sub))):
    manual = sub["stars"].iloc[:k].mean()
    assert abs(sub["u_prior_mean"].iloc[k] - manual) < 1e-9
print("prior-признаки строго по прошлому, ручная сверка прошла")

ручная сверка сходится, средняя по прошлым отзывам юзера совпадает с пересчётом, значит будущее не подсматриваем.

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler
mu_train = ds.loc[ds.split == "train", "stars"].mean()
ds["u_has_hist"] = (ds.u_cnt > 0).astype(float)
ds["b_has_hist"] = (ds.b_cnt > 0).astype(float)
fill = {"u_prior_mean": mu_train, "b_prior_mean": mu_train, "u_days_prev": 3650.0, "b_days_prev": 3650.0, "u_geo_dist": 0.0, "price_aff": 0.0}
ds = ds.fillna(fill)
for c in ["u_cnt", "b_cnt", "u_days_prev", "b_days_prev", "u_geo_dist"]:
    ds[c] = np.log1p(ds[c])
PRIOR_ALL = USER_PRIOR + BIZ_PRIOR + CROSS_PRIOR
prior_scaler = StandardScaler().fit(ds.loc[ds.split == "train", PRIOR_ALL])
ds[PRIOR_ALL] = prior_scaler.transform(ds[PRIOR_ALL])
ds[["review_id"] + PRIOR_ALL].to_parquet(PROCESSED / "task1_prior_feats.parquet", index=False)
joblib.dump(prior_scaler, ARTIFACTS / "task1_prior_scaler.joblib")
len(PRIOR_ALL)

пропуски у новых юзеров и заведений заполнили средним train и большой давностью, тяжёлые счётчики прологарифмировали и отмасштабировали скейлером с fit на train. prior-признаки сохранены отдельным parquet.

## сборка единого фрейма

склеиваем всё по review_id в один DataFrame, таргет, признаки, сырые агрегаты для бейзлайнов и ключи для сегментов лежат в одном порядке строк, позиционных рассинхронов нет.

In [ ]:
import json
emb = pd.read_parquet(PROCESSED / "task1_embed_inputs.parquet")
feats = pd.read_parquet(PROCESSED / "task1_features.parquet", columns=["review_id"] + TIME_COLS)
keycols = ["review_id", "user_id", "business_id", "biz_avg_stars", "average_stars"]
keys = pd.read_parquet(PROCESSED / "task1_dataset.parquet", columns=keycols)
df = emb.merge(feats, on="review_id", validate="one_to_one")
df = df.merge(keys, on="review_id", validate="one_to_one")
df = df.merge(ds[["review_id"] + PRIOR_ALL], on="review_id", validate="one_to_one")
cat_vocab = json.load(open(ARTIFACTS / "task1_vocab_categories.json"))
biz_vocab = json.load(open(ARTIFACTS / "task1_vocab_business.json"))
N_CAT_VOCAB = len(cat_vocab) + 1
N_BIZ_VOCAB = len(biz_vocab) + 1
assert df["city_idx"].max() < N_CITY and df["biz_idx"].max() < N_BIZ_VOCAB
assert not df[USER_NUM + BIZ_NUM + CROSS + TIME_COLS + PRIOR_ALL].isna().any().any()
print(f"строк {len(df):,} | категорий {N_CAT_VOCAB} | заведений {N_BIZ_VOCAB}")
print(df["split"].value_counts().to_dict())

все три источника склеились один к одному, пропусков в признаках нет, индексы города и заведения в пределах словарей.

## тензоры и cold/warm сегменты

готовим тензоры на device, наборы колонок параметризуем, чтобы той же функцией собирать абляции. для теста сразу размечаем холодные и тёплые сегменты, видел ли train этого юзера и это заведение.

In [ ]:
def make_tensors(d, ucols, bcols, ccols):
    t = lambda a, dt: torch.tensor(np.asarray(a), dtype=dt, device=device)
    out = {}
    out["xu"] = t(d[ucols].to_numpy(np.float32), torch.float32)
    out["xb"] = t(d[bcols].to_numpy(np.float32), torch.float32)
    out["ctx"] = t(d[ccols].to_numpy(np.float32), torch.float32)
    out["cats"] = t(d[CATID_COLS].to_numpy(np.int64), torch.long)
    out["city"] = torch.nn.functional.one_hot(t(d["city_idx"].to_numpy(np.int64), torch.long), N_CITY).float()
    out["biz"] = t(d["biz_idx"].to_numpy(np.int64), torch.long)
    out["y"] = t(d["stars"].to_numpy(np.float32), torch.float32)
    return out

In [ ]:
splits = {s: df[df["split"] == s] for s in ["train", "val", "test"]}
y_val = splits["val"]["stars"].to_numpy()
y_te = splits["test"]["stars"].to_numpy()

In [ ]:
train_users = set(splits["train"]["user_id"])
train_biz = set(splits["train"]["business_id"])
warm_user = splits["test"]["user_id"].isin(train_users).to_numpy()
warm_biz = splits["test"]["business_id"].isin(train_biz).to_numpy()
print(f"холодных юзеров {1 - warm_user.mean():.1%}, холодных заведений {1 - warm_biz.mean():.1%}")

в тесте большинство юзеров холодные, заведений холодных меньше. дальше смотрим качество по этим сегментам отдельно.

## бейзлайны

планка, которую сеть обязана перебить. всё считаем на тех же train и test, предсказания клипаем в диапазон от 1 до 5, как и у сети.

In [ ]:
def rmse(y, p):
    return float(np.sqrt(np.mean((y - np.clip(p, 1, 5)) ** 2)))

def mae(y, p):
    return float(np.mean(np.abs(y - np.clip(p, 1, 5))))

In [ ]:
from sklearn.linear_model import Ridge
mu = splits["train"]["stars"].mean()
te = splits["test"]
NUM_ALL = USER_NUM + BIZ_NUM + CROSS + USER_PRIOR + BIZ_PRIOR
ridge = Ridge(alpha=1.0).fit(splits["train"][NUM_ALL], splits["train"]["stars"])
baselines = {
    "глобальное среднее train": np.full(len(te), mu),
    "средняя заведения (biz_avg_stars)": te["biz_avg_stars"].to_numpy(),
    "средняя юзера (average_stars)": te["average_stars"].to_numpy(),
    "bias: user_avg + biz_avg - mu": (te["average_stars"] + te["biz_avg_stars"] - mu).to_numpy(),
    "Ridge на 23 числовых (с prior)": ridge.predict(te[NUM_ALL]),
}
base_table = pd.DataFrame([{"модель": k, "RMSE": rmse(y_te, p), "MAE": mae(y_te, p)} for k, p in baselines.items()])
bias_rmse = base_table.loc[base_table["модель"].str.startswith("bias"), "RMSE"].item()
p_bias = np.clip(te["average_stars"] + te["biz_avg_stars"] - mu, 1, 5).to_numpy()
base_table.round(4)

ridge на всех числовых не лучше суммы двух средних, линейная модель планку не двигает. дальше сеть должна выиграть за счёт нелинейностей и взаимодействий.

## модели

FlatMLP это плоская сеть на полной конкатенации. InteractionMLP это две ветки и явное взаимодействие, скалярное и поэлементное произведение, у неё параметризуются ёмкость, голова и эмбеддинг заведения. DCNv2 это cross-сеть и deep-ствол параллельно над общим входом. категории везде кодируются EmbeddingBag со средним и нулевым паддингом.

In [ ]:
def mlp(dims, p_drop=DROPOUT):
    layers = []
    for i in range(len(dims) - 1):
        layers += [nn.Linear(dims[i], dims[i + 1]), nn.BatchNorm1d(dims[i + 1]), nn.ReLU(), nn.Dropout(p_drop)]
    return nn.Sequential(*layers)

In [ ]:
class FlatMLP(nn.Module):
    def __init__(self, du, db, dc):
        super().__init__()
        self.cat_emb = nn.EmbeddingBag(N_CAT_VOCAB, D_CAT, mode="mean", padding_idx=0)
        self.body = nn.Sequential(mlp([du + db + dc + D_CAT + N_CITY, 256, 128, 64]), nn.Linear(64, 1))

    def forward(self, b):
        x = torch.cat([b["xu"], b["xb"], b["ctx"], self.cat_emb(b["cats"]), b["city"]], dim=1)
        return self.body(x).squeeze(1)

In [ ]:
class InteractionMLP(nn.Module):
    def __init__(self, du, db, dc, use_biz_emb=False, d_cat=D_CAT, d_branch=D_BRANCH, hidden=64, top=(128, 64), out=1):
        super().__init__()
        self.cat_emb = nn.EmbeddingBag(N_CAT_VOCAB, d_cat, mode="mean", padding_idx=0)
        self.use_biz_emb = use_biz_emb
        d_biz_in = db + d_cat + N_CITY
        if use_biz_emb:
            self.biz_emb = nn.Embedding(N_BIZ_VOCAB, D_BIZ_EMB, padding_idx=0)
            d_biz_in += D_BIZ_EMB
        self.user_mlp = mlp([du, hidden, d_branch])
        self.biz_mlp = mlp([d_biz_in, hidden, d_branch])
        self.top = nn.Sequential(mlp([3 * d_branch + 1 + dc, *top]), nn.Linear(top[-1], out))

    def forward(self, b):
        u = self.user_mlp(b["xu"])
        biz_in = [b["xb"], self.cat_emb(b["cats"]), b["city"]]
        if self.use_biz_emb:
            biz_in.append(self.biz_emb(b["biz"]))
        v = self.biz_mlp(torch.cat(biz_in, dim=1))
        had = u * v
        dot = had.sum(dim=1, keepdim=True)
        o = self.top(torch.cat([u, v, had, dot, b["ctx"]], dim=1))
        return o.squeeze(1) if o.shape[1] == 1 else o

In [ ]:
class CrossNet(nn.Module):
    def __init__(self, d, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layers)])

    def forward(self, x0):
        x = x0
        for lin in self.layers:
            x = x0 * lin(x) + x
        return x


class DCNv2(nn.Module):
    def __init__(self, du, db, dc, d_cat=D_CAT, n_cross=3):
        super().__init__()
        self.cat_emb = nn.EmbeddingBag(N_CAT_VOCAB, d_cat, mode="mean", padding_idx=0)
        d_in = du + db + dc + d_cat + N_CITY
        self.cross = CrossNet(d_in, n_cross)
        self.deep = mlp([d_in, 256, 128])
        self.head = nn.Linear(d_in + 128, 1)

    def forward(self, b):
        x = torch.cat([b["xu"], b["xb"], b["ctx"], self.cat_emb(b["cats"]), b["city"]], dim=1)
        return self.head(torch.cat([self.cross(x), self.deep(x)], dim=1)).squeeze(1)

In [ ]:
U_FULL = USER_NUM + USER_PRIOR
B_FULL = BIZ_NUM + BIZ_PRIOR
C_FULL = CROSS + TIME_COLS
STAR_VALUES = torch.arange(1.0, 6.0, device=device)

In [ ]:
def n_params(m):
    return sum(p.numel() for p in m.parameters())

big_head = InteractionMLP(len(U_FULL), len(B_FULL), len(C_FULL), d_cat=32, d_branch=48, hidden=128, top=(256, 128), out=5)
print(f"FlatMLP {n_params(FlatMLP(len(U_FULL), len(B_FULL), len(C_FULL))):,}")
print(f"InteractionMLP {n_params(InteractionMLP(len(U_FULL), len(B_FULL), len(C_FULL))):,}")
print(f"InteractionMLP крупнее, 5 классов {n_params(big_head):,}")
print(f"DCNv2 {n_params(DCNv2(len(U_FULL), len(B_FULL), len(C_FULL))):,}")

базовая сеть небольшая, запас по данным есть, поэтому дальше пробуем увеличить ёмкость.

## цикл обучения

один цикл для всех моделей, AdamW, early stopping по val RMSE, лучшие веса берём с лучшей эпохи. для классификационной головы предсказание это матожидание softmax. test_pred считаем но не печатаем, тест показываем один раз в результатах.

In [ ]:
LOSS_FNS = {"MSE": nn.MSELoss(), "Huber": nn.SmoothL1Loss(beta=1.0), "CE": nn.CrossEntropyLoss()}

def batches(tensors, batch_size, shuffle):
    n = len(tensors["y"])
    idx = torch.randperm(n, device=device) if shuffle else torch.arange(n, device=device)
    for i in range(0, n, batch_size):
        j = idx[i:i + batch_size]
        yield {k: v[j] for k, v in tensors.items()}

In [ ]:
import time

def run_experiment(name, factory, loss_kind, ucols, bcols, ccols, seed=SEED, verbose=False):
    tens = {s: make_tensors(splits[s], ucols, bcols, ccols) for s in ["train", "val", "test"]}
    is_cls = loss_kind == "CE"
    loss_fn = LOSS_FNS[loss_kind]
    torch.manual_seed(seed)
    model = factory(len(ucols), len(bcols), len(ccols)).to(device)

    def decode(out):
        return (torch.softmax(out, dim=1) * STAR_VALUES).sum(dim=1) if is_cls else out

    @torch.no_grad()
    def predict_t(tensors):
        model.eval()
        preds = []
        for b in batches(tensors, 8192, shuffle=False):
            o = decode(model({k: v for k, v in b.items() if k != "y"}))
            preds.append(o.float().cpu().numpy())
        return np.concatenate(preds)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    best = {"rmse": np.inf, "state": None, "epoch": -1}
    history = []
    t0 = time.time()
    for epoch in range(MAX_EPOCHS):
        model.train()
        tot, nb = 0.0, 0
        for b in batches(tens["train"], BATCH, shuffle=True):
            if len(b["y"]) < 2:
                continue
            opt.zero_grad()
            out = model({k: v for k, v in b.items() if k != "y"})
            target = (b["y"] - 1).long() if is_cls else b["y"]
            loss = loss_fn(out, target)
            loss.backward()
            opt.step()
            tot, nb = tot + loss.item(), nb + 1
        val_rmse = rmse(y_val, predict_t(tens["val"]))
        assert np.isfinite(val_rmse)
        sched.step(val_rmse)
        history.append({"epoch": epoch, "train_loss": tot / nb, "val_rmse": val_rmse})
        if val_rmse < best["rmse"]:
            state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            best = {"rmse": val_rmse, "state": state, "epoch": epoch}
        if verbose:
            print(f"{name} эпоха {epoch} train {tot / nb} val {val_rmse}")
        if epoch - best["epoch"] >= PATIENCE:
            break
    model.load_state_dict(best["state"])
    res = {"name": name, "val_rmse": best["rmse"], "history": pd.DataFrame(history), "state": best["state"]}
    res["val_pred"] = predict_t(tens["val"])
    res["test_pred"] = predict_t(tens["test"])
    print(f"{name} val {best['rmse']} {time.time() - t0}c")
    return res

## обучаем сетку

две архитектуры на два лосса на полном наборе признаков. сравнение архитектур делаем при фиксированном лоссе.

In [ ]:
arch_list = [("плоский", lambda du, db, dc: FlatMLP(du, db, dc)), ("взаимодействие", lambda du, db, dc: InteractionMLP(du, db, dc))]
runs = {}
for arch_name, factory in arch_list:
    for loss_kind in ["MSE", "Huber"]:
        key = f"MLP {arch_name} ({loss_kind})"
        runs[key] = run_experiment(key, factory, loss_kind, U_FULL, B_FULL, C_FULL)

четыре конфигурации обучились, по val видно общий уровень, выбор лучшей делаем дальше.